# Regional TCVIS sample 2007-2026 — COG + Zarr

Small-AOI counterpart to `Production_2005-2024.ipynb`. The trend maths is the production
path (`modules/high_level_functions_CO2.runTCTrend`), wrapped by
`modules.regional_trend.RegionalTrend` so the same few lines scale to another AOI or window.

* window: **2007-2026**, July-August, `CLOUD_COVER < 70`
* AOI: lat **70.7-70.9 N**, lon **154.0-153.5 W**
* output: **COG** (EPSG:3413, 30 m) + **Zarr**, written outside the repo
* product: the same **3-band uint8** image the production notebook exports with
  `ee.Image(trend['visual']).toByte()` (`unitScale(-0.12, 0.12) * 255`)

Compare with `notebooks/Trend_Calculation_v2.ipynb`: same idea, but that one is L8/L9 only,
has no sigma outlier mask, and exports at 70 m — its numbers do not match the product.

## Parameters and wrapper

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
OUT = REPO.parent / "out"          # sibling of the repo -> stays out of git
OUT.mkdir(exist_ok=True)

from modules.regional_trend import RegionalTrend

run = RegionalTrend(
    lon=(-154.0, -153.5),          # 154.0W - 153.5W
    lat=(70.7, 70.9),
    years=(2007, 2026),
    start_month=7,
    end_month=8,
    max_cloud=70,
    scale=30,
    crs="EPSG:3413",
    project="pdg-wg-workflow",
)
run.describe()

## Build the trend image (server-side)

L5/L7/L8/L9 are merged inside `runTCTrend`; slopes are `TCB/TCG/TCW` linear fits over the
annual median mosaics. `run.visual_image()` is the uint8 product, `run.slopes_image()` the
float one.

In [ ]:
trend = run.trend()
print("float bands:", trend["data"].bandNames().getInfo())
print("visual bands:", trend["visual"].bandNames().getInfo())

## Pull into xarray

One request on the final grid, so the UTM -> polar-stereographic conversion happens once, bilinearly.

In [ ]:
ds = run.dataset()                 # kind="visual" by default
print("grid:", dict(ds.sizes), "| crs:", ds.attrs["crs"], "| product:", ds.attrs["product"])
print("dtype:", {b: str(ds[b].dtype) for b in ds.data_vars})
ds

## Write COG + Zarr and verify

In [ ]:
cog = run.to_cog(OUT / "regional_sample_2007-2026_visual.tif")
nobs = run.to_cog(OUT / "regional_sample_2007-2026_nobs.tif", kind="observations")
zarr_store = run.to_zarr(OUT / "regional_sample_2007-2026_visual.zarr")

checks = run.verify(cog)
checks

`verify` proves the delivered file is the right shape: EPSG:3413, uint8, the three slope
bands, a real internal mask (`mask_values == [0, 255]`) and coverage of the AOI.

## Cross-check against Earth Engine's own statistics

In [ ]:
import numpy as np

ee_stats = run.ee_stats()          # same uint8 image, reduced on EE
rows = {}
for band in ds.data_vars:
    v = ds[band].values
    v = v[~np.isnan(v)]
    ee_mean = ee_stats["mean"][band]
    rows[band] = dict(
        cog_mean=round(float(v.mean()), 5),
        ee_mean=round(ee_mean, 5),
        rel_diff=round(abs(v.mean() - ee_mean) / max(abs(ee_mean), 1e-6), 4),
        valid_percent=round(100 * v.size / ds[band].size, 1),
    )
rows

## Quicklook

uint8 scale: 0 = slope -0.12, 128 = no trend, 255 = +0.12.

In [ ]:
png = run.quicklook(OUT / "regional_sample_2007-2026.png")
from IPython.display import Image, display
display(Image(filename=str(png)))

## The float variant

For analysis rather than display, the raw trend is one argument away — same fetch path,
float32, `TC slope x 10`, NaN where masked:

```python
float_ds = run.dataset("slopes")
run.to_cog(OUT / "regional_sample_2007-2026_slopes.tif", kind="slopes")
```

## Reading the result

Early years are thin and that is visible in the data, not hidden: 2007-2012 rests on L5 + L7
only (4-12 usable Jul-Aug scenes per year in this patch, 2011 the weakest), and L7 is SLC-off,
so those medians carry striping the 2 sigma filter does not repair. The `_nobs.tif` band records
per-pixel observation counts — filter on it rather than moving the start year.